In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  01_data_download.ipynb — DriftFire                        ║
# ║  Copy each CELL block into a separate Jupyter cell.        ║
# ║  Run them in order, top to bottom. 5 cells total.          ║
# ╚══════════════════════════════════════════════════════════════╝
 
 
# ============================================================
# CELL 1 — Install Packages (subprocess method)
# ============================================================
# WHY subprocess: Your OneDrive project path has spaces, which
# breaks !pip install. subprocess.check_call routes through
# the venv's Python binary directly, avoiding the shell parse.
#
# WHAT YOU SHOULD SEE: Each package prints "Done." then
# "All packages installed successfully!" at the end. ~30-45s.
# ============================================================
 
import subprocess
import sys
 
packages = [
    "yfinance",
    "pandas",
    "numpy",
    "matplotlib",
    "seaborn",
    "pyarrow",
    "scipy",
    "statsmodels",
    "finviz",
    "lxml",          # needed for pd.read_html (Wikipedia scrape)
]
 
for package in packages:
    print(f"Installing {package}...")
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", package, "--quiet"]
    )
    print(f"Done.")
 
print("\nAll packages installed successfully!")
 

Installing yfinance...
Done.
Installing pandas...
Done.
Installing numpy...
Done.
Installing matplotlib...
Done.
Installing seaborn...
Done.
Installing pyarrow...
Done.
Installing scipy...
Done.
Installing statsmodels...
Done.
Installing finviz...
Done.
Installing lxml...
Done.

All packages installed successfully!


In [9]:
# ============================================================
# CELL 2 — Imports, Config & Path Setup
# ============================================================
# WHAT YOU SHOULD SEE:
#   Libraries loaded successfully
#   yfinance version: 1.2.0
#   pandas version: 2.3.3
#   Project root: /Users/aaravaguru/.../Quant Project 1 - DriftFire
#   Raw data dir created ✓
#   Processed dir created ✓
#
# NOTE: The NotOpenSSLWarning from urllib3 is harmless — the
# warnings filter suppresses it so it won't clutter your output.
# ============================================================
 
import warnings
warnings.filterwarnings("ignore")
 
import yfinance as yf
import pandas as pd
import numpy as np
import os
from pathlib import Path
 
print("Libraries loaded successfully")
print(f"yfinance version: {yf.__version__}")
print(f"pandas version: {pd.__version__}")
 
# --- project paths ---
# Path.cwd() = the notebooks/ folder when you run from VSCode
PROJECT_ROOT = Path.cwd().parent
RAW_DIR      = PROJECT_ROOT / "data" / "raw"
PROC_DIR     = PROJECT_ROOT / "data" / "processed"
 
RAW_DIR.mkdir(parents=True, exist_ok=True)
PROC_DIR.mkdir(parents=True, exist_ok=True)
 
print(f"\nProject root   : {PROJECT_ROOT}")
print(f"Raw data dir   : {RAW_DIR}  ✓")
print(f"Processed dir  : {PROC_DIR}  ✓")
 
# --- constants used by all downstream cells ---
START = "2018-01-01"
END   = "2024-12-31"
 
SECTOR_ETFS = [
    "XLK", "XLC", "XLE", "XLF", "XLV",
    "XLI", "XLP", "XLY", "XLRE", "XLB", "XLU"
]
REGIME_TICKER = "QQQ"
 

Libraries loaded successfully
yfinance version: 1.2.0
pandas version: 2.3.3

Project root   : /Users/aaravaguru/Library/CloudStorage/OneDrive-IndianaUniversity/Quant Project 1 - DriftFIre
Raw data dir   : /Users/aaravaguru/Library/CloudStorage/OneDrive-IndianaUniversity/Quant Project 1 - DriftFIre/data/raw  ✓
Processed dir  : /Users/aaravaguru/Library/CloudStorage/OneDrive-IndianaUniversity/Quant Project 1 - DriftFIre/data/processed  ✓


In [10]:
# ============================================================
# CELL 3 — Build Ticker Universe & Download OHLCV
# ============================================================
# FIX: Wikipedia blocks Python's default user-agent (403 error).
# Using a hardcoded S&P 500 list instead — more reliable and
# means the notebook doesn't break if Wikipedia changes layout.
# ============================================================
 
sp500_tickers = [
    "A", "AAL", "AAPL", "ABBV", "ABNB", "ABT", "ACGL", "ACM", "ACN", "ADBE",
    "ADI", "ADM", "ADP", "ADSK", "AEE", "AEP", "AES", "AFL", "AIG", "AIZ",
    "AJG", "AKAM", "ALB", "ALGN", "ALL", "ALLE", "AMAT", "AMCR", "AMD", "AME",
    "AMGN", "AMP", "AMT", "AMZN", "ANET", "ANSS", "AON", "AOS", "APA", "APD",
    "APH", "APTV", "ARE", "ATO", "ATVI", "AVGO", "AVY", "AWK", "AXP", "AZO",
    "BA", "BAC", "BAX", "BBWI", "BBY", "BDX", "BEN", "BF-B", "BG", "BIIB",
    "BIO", "BK", "BKNG", "BKR", "BLK", "BMY", "BR", "BRK-B", "BRO", "BSX",
    "BWA", "BXP", "C", "CAG", "CAH", "CARR", "CAT", "CB", "CBOE", "CBRE",
    "CCI", "CCL", "CDAY", "CDNS", "CDW", "CE", "CEG", "CF", "CFG", "CHD",
    "CHRW", "CHTR", "CI", "CINF", "CL", "CLX", "CMA", "CMCSA", "CME", "CMG",
    "CMI", "CMS", "CNC", "CNP", "COF", "COO", "COP", "COST", "CPB", "CPRT",
    "CPT", "CRL", "CRM", "CSCO", "CSGP", "CSX", "CTAS", "CTLT", "CTRA", "CTSH",
    "CTVA", "CVS", "CVX", "CZR", "D", "DAL", "DD", "DE", "DECK", "DFS",
    "DG", "DGX", "DHI", "DHR", "DIS", "DISH", "DLR", "DLTR", "DOV", "DOW",
    "DPZ", "DRI", "DTE", "DUK", "DVA", "DVN", "DXC", "DXCM", "EA", "EBAY",
    "ECL", "ED", "EFX", "EIX", "EL", "EMN", "EMR", "ENPH", "EOG", "EPAM",
    "EQIX", "EQR", "EQT", "ES", "ESS", "ETN", "ETR", "ETSY", "EVRG", "EW",
    "EXC", "EXPD", "EXPE", "EXR", "F", "FANG", "FAST", "FBHS", "FCX", "FDS",
    "FDX", "FE", "FFIV", "FIS", "FISV", "FITB", "FLT", "FMC", "FOX", "FOXA",
    "FRC", "FRT", "FSLR", "FTNT", "FTV", "GD", "GE", "GILD", "GIS", "GL",
    "GLW", "GM", "GNRC", "GOOG", "GOOGL", "GPC", "GPN", "GRMN", "GS", "GWW",
    "HAL", "HAS", "HBAN", "HCA", "HOLX", "HON", "HPE", "HPQ", "HRL",
    "HSIC", "HST", "HSY", "HUM", "HWM", "IBM", "ICE", "IDXX", "IEX", "IFF",
    "ILMN", "INCY", "INTC", "INTU", "INVH", "IP", "IPG", "IQV", "IR", "IRM",
    "ISRG", "IT", "ITW", "IVZ", "J", "JBHT", "JCI", "JKHY", "JNJ", "JNPR",
    "JPM", "K", "KDP", "KEY", "KEYS", "KHC", "KIM", "KLAC", "KMB", "KMI",
    "KMX", "KO", "KR", "L", "LDOS", "LEN", "LH", "LHX", "LIN", "LKQ",
    "LLY", "LMT", "LNC", "LNT", "LOW", "LRCX", "LUMN", "LUV", "LVS", "LW",
    "LYB", "LYV", "MA", "MAA", "MAR", "MAS", "MCD", "MCHP", "MCK", "MCO",
    "MDLZ", "MDT", "MET", "META", "MGM", "MHK", "MKC", "MKTX", "MLM", "MMC",
    "MMM", "MNST", "MO", "MOH", "MOS", "MPC", "MPWR", "MRK", "MRNA", "MRO",
    "MS", "MSCI", "MSFT", "MSI", "MTB", "MTCH", "MTD", "MU", "NCLH", "NDAQ",
    "NDSN", "NEE", "NEM", "NFLX", "NI", "NKE", "NOC", "NOW", "NRG", "NSC",
    "NTAP", "NTRS", "NUE", "NVDA", "NVR", "NWL", "NWS", "NWSA", "NXPI", "O",
    "ODFL", "OGN", "OKE", "OMC", "ON", "ORCL", "ORLY", "OTIS", "OXY", "PARA",
    "PAYC", "PAYX", "PCAR", "PCG", "PEAK", "PEG", "PEP", "PFE", "PFG", "PG",
    "PGR", "PH", "PHM", "PKG", "PKI", "PLD", "PM", "PNC", "PNR", "PNW",
    "POOL", "PPG", "PPL", "PRU", "PSA", "PSX", "PTC", "PVH", "PWR", "PXD",
    "PYPL", "QCOM", "QRVO", "RCL", "RE", "REG", "REGN", "RF", "RHI", "RJF",
    "RL", "RMD", "ROK", "ROL", "ROP", "ROST", "RSG", "RTX", "RVTY", "SBAC",
    "SBNY", "SBUX", "SCHW", "SEE", "SHW", "SIVB", "SJM", "SLB", "SNA", "SNPS",
    "SO", "SPG", "SPGI", "SRE", "STE", "STT", "STX", "STZ", "SWK", "SWKS",
    "SYF", "SYK", "SYY", "T", "TAP", "TDG", "TDY", "TECH", "TEL", "TER",
    "TFC", "TFX", "TGT", "TMO", "TMUS", "TPR", "TRGP", "TRMB", "TROW", "TRV",
    "TSCO", "TSLA", "TSN", "TT", "TTWO", "TXN", "TXT", "TYL", "UAL", "UDR",
    "UHS", "ULTA", "UNH", "UNP", "UPS", "URI", "USB", "V", "VFC", "VICI",
    "VLO", "VMC", "VRSK", "VRSN", "VRTX", "VTR", "VTRS", "VZ", "WAB", "WAT",
    "WBA", "WBD", "WDC", "WEC", "WELL", "WFC", "WHR", "WM", "WMB", "WMT",
    "WRB", "WRK", "WST", "WTW", "WY", "WYNN", "XEL", "XOM", "XRAY", "XYL",
    "YUM", "ZBH", "ZBRA", "ZION", "ZTS",
]
 
# NOTE: This list includes some tickers that were delisted or removed
# from the S&P 500 between 2018-2024 (ATVI, FRC, SIVB, SBNY, etc.).
# yfinance will simply skip any it can't find — no errors, just fewer rows.
# This actually *helps* with survivorship bias since we're keeping the
# delisted names in the download attempt.
 
# --- Combine with ETFs ---
all_tickers = sorted(set(sp500_tickers + SECTOR_ETFS + [REGIME_TICKER]))
 
n_stocks = len([t for t in all_tickers if t not in SECTOR_ETFS + [REGIME_TICKER]])
n_etfs   = len([t for t in all_tickers if t in SECTOR_ETFS + [REGIME_TICKER]])
print(f"Universe: {len(all_tickers)} tickers ({n_stocks} stocks + {n_etfs} ETFs)")
print(f"Date range: {START} → {END}")
print(f"\nDownloading... (this takes 3-8 min)\n")
 
# --- Download via yfinance ---
raw = yf.download(
    tickers     = all_tickers,
    start       = START,
    end         = END,
    group_by    = "ticker",
    auto_adjust = True,
    threads     = True,
)
 
print(f"\nRaw download shape: {raw.shape}")
print(f"Column levels: {raw.columns.nlevels}")
print(f"Date range: {raw.index.min().date()} → {raw.index.max().date()}")
 
# --- Save raw ---
raw.to_parquet(RAW_DIR / "universe_raw.parquet")
print(f"\nSaved → {RAW_DIR / 'universe_raw.parquet'}")
 

Universe: 506 tickers (494 stocks + 12 ETFs)
Date range: 2018-01-01 → 2024-12-31

Downloading... (this takes 3-8 min)



[                       1%                       ]  5 of 506 completed$CTLT: possibly delisted; no timezone found
[*********             18%                       ]  93 of 506 completed$WRK: possibly delisted; no timezone found
[**********            20%                       ]  99 of 506 completed$SIVB: possibly delisted; no timezone found
[**********            21%                       ]  107 of 506 completed$ATVI: possibly delisted; no timezone found
[***********           23%                       ]  116 of 506 completed$FLT: possibly delisted; no timezone found
[************          24%                       ]  119 of 506 completed$CDAY: possibly delisted; no timezone found
[*******************   39%                       ]  196 of 506 completed$DFS: possibly delisted; no timezone found
[********************  42%                       ]  212 of 506 completed$DISH: possibly delisted; no timezone found
[**********************45%                       ]  228 of 506 completed$IPG: p


Raw download shape: (1760, 2552)
Column levels: 2
Date range: 2018-01-02 → 2024-12-30

Saved → /Users/aaravaguru/Library/CloudStorage/OneDrive-IndianaUniversity/Quant Project 1 - DriftFIre/data/raw/universe_raw.parquet


In [12]:


# ============================================================
# CELL 4 — Reshape to Long Format & Apply Daily Screener
# ============================================================
# WHAT THIS DOES:
#   1. Loads raw parquet, converts wide MultiIndex → clean long.
#   2. Computes technical indicators per ticker per day:
#        EMA(8), EMA(21), EMA(50), avg vol 10D/20D,
#        daily % change, dollar volume (cap proxy).
#   3. Applies your TradingView screener filters daily:
#        ✓ Close > $3
#        ✓ Daily change > 0%
#        ✓ Dollar volume > $1.5M (proxy for mkt cap > $300M)
#        ✓ Avg volume 10D > 500,000
#        ✓ Close > EMA(21)
#        ✓ Close > EMA(50)
#   4. Stamps screener_pass = True/False on every row.
#
# WHAT YOU SHOULD SEE:
#   Long format row count, indicator computation progress,
#   then a daily screener pass summary showing avg/min/max
#   tickers passing per day + sample of last 10 days.
#   Expect ~100-250 avg in bull regimes, near 0 in drawdowns.
#
# NO LOOKAHEAD BIAS:
#   All indicators are shifted by 1 day before the screener
#   evaluates them. Day T's screener uses day T-1 close.
#   Entry would happen at day T's open.
# ============================================================
 
# --- 1. Load raw and reshape wide → long ---
raw = pd.read_parquet(RAW_DIR / "universe_raw.parquet")
 
records = []
 
if isinstance(raw.columns, pd.MultiIndex):
    tickers_in_data = raw.columns.get_level_values(0).unique().tolist()
 
    for ticker in tickers_in_data:
        try:
            sub = raw[ticker].copy()
        except KeyError:
            continue
 
        sub.columns = sub.columns.str.lower().str.strip()
 
        required = {"open", "high", "low", "close", "volume"}
        if not required.issubset(set(sub.columns)):
            continue
 
        sub = sub[["open", "high", "low", "close", "volume"]].copy()
        sub["ticker"] = ticker
        sub.index.name = "date"
        records.append(sub.reset_index())
 
    long = pd.concat(records, ignore_index=True)
else:
    long = raw.reset_index()
    long.columns = [c.lower() for c in long.columns]
 
long["date"] = pd.to_datetime(long["date"])
long = long.dropna(subset=["close", "volume"])
long = long[long["volume"] > 0]
long = long.sort_values(["ticker", "date"]).reset_index(drop=True)
 
print(f"Long format: {long.shape[0]:,} rows × {long.shape[1]} cols")
print(f"Unique tickers: {long['ticker'].nunique()}")
print(f"Trading days: {long['date'].nunique()}")
 
# --- 2. Tag ETFs vs stocks ---
etf_set = set(SECTOR_ETFS + [REGIME_TICKER])
long["is_etf"] = long["ticker"].isin(etf_set)
 
# --- 3. Compute indicators per ticker ---
def compute_indicators(group):
    g = group.sort_values("date").copy()
 
    # EMAs
    g["ema8"]  = g["close"].ewm(span=8,  adjust=False).mean()
    g["ema21"] = g["close"].ewm(span=21, adjust=False).mean()
    g["ema50"] = g["close"].ewm(span=50, adjust=False).mean()
 
    # Volume indicators
    g["avg_vol_10d"] = g["volume"].rolling(10, min_periods=10).mean()
    g["avg_vol_20d"] = g["volume"].rolling(20, min_periods=20).mean()
 
    # Daily change
    g["daily_chg_pct"] = g["close"].pct_change() * 100
 
    # Dollar volume (market cap proxy)
    g["dollar_volume"] = g["close"] * g["volume"]
 
    return g
 
print("\nComputing indicators...")
long = long.groupby("ticker", group_keys=False).apply(compute_indicators)
long = long.sort_values(["ticker", "date"]).reset_index(drop=True)
print(f"After indicators: {long.shape[0]:,} rows × {long.shape[1]} cols")
 
# --- 4. Apply screener on SHIFTED (prior-day) values ---
stocks = long[~long["is_etf"]].copy()
 
shift_cols = [
    "close", "ema8", "ema21", "ema50",
    "avg_vol_10d", "avg_vol_20d",
    "daily_chg_pct", "dollar_volume"
]
 
for col in shift_cols:
    stocks[f"prev_{col}"] = stocks.groupby("ticker")[col].shift(1)
 
# The 6 TradingView filters, evaluated on prior-day data
stocks["screener_pass"] = (
    (stocks["prev_close"] > 3)                         # price > $3
    & (stocks["prev_daily_chg_pct"] > 0)               # green day
    & (stocks["prev_dollar_volume"] > 1_500_000)       # cap proxy
    & (stocks["prev_avg_vol_10d"] > 500_000)           # liquid
    & (stocks["prev_close"] > stocks["prev_ema21"])    # above 21 EMA
    & (stocks["prev_close"] > stocks["prev_ema50"])    # above 50 EMA
)
 
# --- 5. Daily screener count (screenshot this) ---
daily_count = (
    stocks[stocks["screener_pass"]]
    .groupby("date")["ticker"]
    .nunique()
    .rename("tickers_passing")
)
 
print(f"\n{'='*55}")
print(f"  DAILY SCREENER PASS COUNT")
print(f"{'='*55}")
print(f"  Avg tickers passing per day : {daily_count.mean():.0f}")
print(f"  Min                         : {daily_count.min()}")
print(f"  Max                         : {daily_count.max()}")
print(f"  Median                      : {daily_count.median():.0f}")
print(f"{'='*55}")
print(f"\n  Last 10 trading days:")
print(daily_count.tail(10).to_string())
 
# --- 6. Merge ETFs back in (they don't get screened) ---
etfs = long[long["is_etf"]].copy()
etfs["screener_pass"] = False
for col in shift_cols:
    etfs[f"prev_{col}"] = etfs.groupby("ticker")[col].shift(1)
 
full = pd.concat([stocks, etfs], ignore_index=True)
full = full.sort_values(["ticker", "date"]).reset_index(drop=True)
 
print(f"\nFull dataset: {full.shape[0]:,} rows × {full.shape[1]} cols")
print(f"Stocks passing screener at least once: "
      f"{stocks[stocks['screener_pass']]['ticker'].nunique()}")
 

Long format: 844,614 rows × 7 cols
Unique tickers: 484
Trading days: 1760

Computing indicators...
After indicators: 844,614 rows × 15 cols

  DAILY SCREENER PASS COUNT
  Avg tickers passing per day : 140
  Min                         : 1
  Max                         : 393
  Median                      : 137

  Last 10 trading days:
date
2024-12-16    47
2024-12-17    78
2024-12-18    35
2024-12-19     3
2024-12-20    23
2024-12-23    55
2024-12-24    35
2024-12-26    99
2024-12-27    73
2024-12-30    10

Full dataset: 844,614 rows × 24 cols
Stocks passing screener at least once: 469


In [13]:
# ============================================================
# CELL 5 — Save to Processed Parquet
# ============================================================
# WHAT THIS DOES:
#   Saves the complete dataset (stocks + ETFs, all indicators,
#   screener_pass flag) to data/processed/clean.parquet.
#
# WHAT YOU SHOULD SEE:
#   File path, size in MB, full schema printout.
#   This file is the input for 02_cleaning_qa.ipynb.
# ============================================================
 
out_path = PROC_DIR / "clean.parquet"
full.to_parquet(out_path, index=False)
 
size_mb = out_path.stat().st_size / (1024 * 1024)
 
print(f"Saved → {out_path}")
print(f"File size: {size_mb:.1f} MB")
print(f"\nSchema:")
print(full.dtypes.to_string())
print(f"\nTotal rows   : {full.shape[0]:,}")
print(f"Total columns: {full.shape[1]}")
print(f"\n✅ 01_data_download.ipynb complete.")
print(f"Next step → 02_cleaning_qa.ipynb")
 
 

Saved → /Users/aaravaguru/Library/CloudStorage/OneDrive-IndianaUniversity/Quant Project 1 - DriftFIre/data/processed/clean.parquet
File size: 118.5 MB

Schema:
Price
date                  datetime64[ns]
open                         float64
high                         float64
low                          float64
close                        float64
volume                       float64
ticker                        object
is_etf                          bool
ema8                         float64
ema21                        float64
ema50                        float64
avg_vol_10d                  float64
avg_vol_20d                  float64
daily_chg_pct                float64
dollar_volume                float64
prev_close                   float64
prev_ema8                    float64
prev_ema21                   float64
prev_ema50                   float64
prev_avg_vol_10d             float64
prev_avg_vol_20d             float64
prev_daily_chg_pct           float64
prev_dollar_volume  